In [1]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

print("🚀 [1/4] 环境初始化与代码拉取...")
%cd /kaggle/working
!rm -rf ThermoRG-NN

# 提取密钥并克隆代码
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url}
%cd /kaggle/working/ThermoRG-NN

print("\n🔧 [2/4] 挂载 ThermoRG 物理引擎...")
!pip install -e .
sys.path.insert(0, '/kaggle/working/ThermoRG-NN/src')
sys.path.insert(0, '/kaggle/working/ThermoRG-NN')

# ✅ 修正架构名称（与 constants.py 中 ALL_SPECS 保持一致）
architectures = [
    "ThermoNet-3", "ThermoNet-5", "ThermoNet-7", "ThermoNet-9",
    "ThermoBot-3", "ThermoBot-5", "ThermoBot-7", "ThermoBot-9",
    "ReLUFurnace-3", "ReLUFurnace-5", "ReLUFurnace-7", "ReLUFurnace-9",
    "ResNet-18-CIFAR", "VGG-11-CIFAR", "DenseNet-40-CIFAR"
]
for arch in architectures:
    os.makedirs(f"experiments/lift_test/results/{arch}", exist_ok=True)

print("\n🔥 [3/4] 启动智能断点续传炼丹炉...")
# 配置 Git 身份
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
!git remote set-url origin {repo_url}

try:
    from experiments.lift_test import train
    train.train_phase_a()
    print("\n🎉 全部架构训练圆满结束！")
except Exception as e:
    print(f"\n🚨 训练中途触发异常: {e}")
    print("⚠️ 别慌！已经触发容灾机制，准备抢救已完成的数据...")
finally:
    print("\n📦 [4/4] 无论成败，执行终极抢救性数据推送...")
    # ✅ 修复：用引号包裹 glob pattern，确保 git add 能正确匹配
    commands = [
        'git add "experiments/lift_test/results/*/*.csv"',
        'git add "experiments/lift_test/results/*/*.json"',
        'git commit -m "🚀 Data: Incremental save from Kaggle via Auto-Resume Script"',
        'git push -u origin main'
    ]
    for cmd in commands:
        # ✅ 修复：result 缩进进 for 循环
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/kaggle/working/ThermoRG-NN")
        if result.returncode != 0 and "nothing to commit" not in result.stdout and "nothing to commit" not in result.stderr:
            print(f"❌ 推送警告: {result.stderr.replace(gh_token, '***')}")
        else:
            print(f"✅ 执行成功: {cmd.split()[1]}")

    print("\n🏆 抢救性闭环完成！云端机器即将关机。")

🚀 [1/4] 环境初始化与代码拉取...
/kaggle/working
Cloning into 'ThermoRG-NN'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 202 (delta 37), reused 182 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 265.45 KiB | 8.56 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/kaggle/working/ThermoRG-NN

🔧 [2/4] 挂载 ThermoRG 物理引擎...
Obtaining file:///kaggle/working/ThermoRG-NN
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for thermorg (pyproject.toml) ... done
  Created wheel for thermorg: filename=thermorg-0.1.0-0.editable-py3-none-any.whl size=7135 sha256=26bed1e63ffc7f12c70fb649b0f21a3d0ec1a0a2afed9ace37cc5a32c6b02d6e
  Stored in directory: /tmp/pip-ephem-wheel-cache-dycx3md6/wheels/42/f6/c7/63209

2026-03-30 20:24:47.207842: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774902287.448884      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774902287.509966      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774902288.066465      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774902288.066506      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774902288.066509      24 computation_placer.cc:177] computation placer alr

Phase A: Training all architectures for 30 epochs
Device: cuda


100%|██████████| 170M/170M [00:04<00:00, 34.9MB/s]



Training ThermoNet-3


ThermoNet-3: 100%|██████████| 30/30 [10:11<00:00, 20.39s/it]



🚨 训练中途触发异常: name 'csv' is not defined
⚠️ 别慌！已经触发容灾机制，准备抢救已完成的数据...

📦 [4/4] 无论成败，执行终极抢救性数据推送...
❌ 推送警告: fatal: pathspec 'experiments/lift_test/results/*/*.csv' did not match any files

❌ 推送警告: fatal: pathspec 'experiments/lift_test/results/*/*.json' did not match any files

❌ 推送警告: 
✅ 执行成功: push

🏆 抢救性闭环完成！云端机器即将关机。
